### Authenticate for access token 

Run the cell below, then type your XBRL US Web account email, account password, Client ID, and secret as noted, *pressing the Enter key on the keyboard after each entry*.

In [2]:

print('Enter your XBRL US Web account email: ')
import os, re, sys, json
import requests
import pandas as pd
import numpy as np
import getpass
email = input()
print('Account password: ')
password = getpass.getpass()
print('Client ID: ')
clientid = getpass.getpass()
print('Client secret: ')
secret = getpass.getpass()

body_auth = {'username' : ''.join(email), 
            'client_id': ''.join(clientid), 
            'client_secret' : ''.join(secret), 
            'password' : ''.join(password), 
            'grant_type' : 'password', 
            'platform' : 'ipynb' }
endpoint_auth = 'https://api.xbrl.us/oauth2/token'
res = requests.post(endpoint_auth, data=body_auth)
auth_json = res.json()

if 'error' in auth_json:
    print ("\n\nThere was a problem generating an access token with these credentials. Run the first cell again to enter credentials.")
else:
    print ("\n\nYour access token expires in 60 minutes. After it expires, run the cell immediately below this one to generate a new token and continue to use the query cell. \n\nFor now, skip ahead to the section 'Make a Query'.")
access_token = auth_json['access_token']
refresh_token = auth_json['refresh_token']
newaccess = ''
newrefresh = ''
#print('access token: ' + access_token + ' refresh token: ' + refresh_token)

Enter your XBRL US Web account email: 
Account password: 
Client ID: 
Client secret: 


Your access token expires in 60 minutes. After it expires, run the cell immediately below this one to generate a new token and continue to use the query cell. 

For now, skip ahead to the section 'Make a Query'.


The cell below is only needed to refresh an expired access token after 60 minutes. When the access token no longer returns results, run the cell below to refresh the access token or re-enter credentials by running the cell above. Until the refresh token process is needed, skip ahead to **Make a Query**.

In [ ]:
# #### Refresh token 
# The cell below is only needed to refresh an expired access token after 60 minutes. When the access token no longer returns results, run the cell below to refresh the access token or re-enter credentials by running the cell above. Until the refresh token process is needed, skip ahead to **Make a Query**. 
# 

# In[10]:


token = token if newrefresh != '' else refresh_token 

refresh_auth = {'client_id': ''.join(clientid), 
            'client_secret' : ''.join(secret), 
            'grant_type' : 'refresh_token', 
            'platform' : 'ipynb', 
            'refresh_token' : ''.join(token) }
refreshres = requests.post(endpoint_auth, data=refresh_auth)
refresh_json = refreshres.json()
access_token = refresh_json['access_token']
refresh_token = refresh_json['refresh_token']#print('access token: ' + access_token + 'refresh token: ' + refresh_token)
print('Your access token is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.')
print(access_token)


After the access token confirmation appears above, you can modify the query below, then use the **_Cell >> Run All Below_** menu option from the cell **immediately below this text** to run the entire query for results. 

**Once the initial data query is executed**, use *_Run All Below_* menu option at the highest point 

Refer to XBRL API documentation at https://xbrlus.github.io/xbrl-api/#/Facts/getFactDetails for other endpoints and parameters to filter and return. 

Non-members might not be able to return all data for a query - consider joining XBRL US for comprehensive access - https://xbrl.us/join.

In [41]:
# #### Define the fields to be returned

queried_count = 0
res_df = [] #json.dumps({})


offset_value = 0
#{'offset': int(queried_count)
 #               }

fields = ['period.fiscal-year.sort(DESC)',
         'entity.name.sort(ASC)',
         'concept.local-name.sort(ASC)',
         'fact.value',
         'unit',
         'fact.decimals',
         'report.filing-date'
         ]


# #### Define the parameters of the query, execute search


XBRL_Elements = ['EffectiveIncomeTaxRateReconciliationAtFederalStatutoryIncomeTaxRate',
'EffectiveIncomeTaxRateReconciliationTaxCutsAndJobsActOf2017Percent',
'EffectiveIncomeTaxRateReconciliationTaxCutsAndJobsActOf2017TransitionTaxOnAccumulatedForeignEarningsPercent',
'EffectiveIncomeTaxRateReconciliationStateAndLocalIncomeTaxes',
'EffectiveIncomeTaxRateReconciliationForeignIncomeTaxRateDifferential',
'EffectiveIncomeTaxRateReconciliationTaxCredits',
'EffectiveIncomeTaxRateReconciliationChangeInEnactedTaxRate',
'EffectiveIncomeTaxRateReconciliationChangeInDeferredTaxAssetsValuationAllowance',
'EffectiveIncomeTaxRateReconciliationShareBasedCompensationExcessTaxBenefitPercent',
'EffectiveIncomeTaxRateReconciliationOtherAdjustments',
'EffectiveIncomeTaxRateReconciliationOtherReconcilingItemsPercent',
'EffectiveIncomeTaxRateContinuingOperations',
'IncomeTaxReconciliationIncomeTaxExpenseBenefitAtFederalStatutoryIncomeTaxRate',
'EffectiveIncomeTaxRateReconciliationTaxCutsAndJobsActOf2017Amount',
'EffectiveIncomeTaxRateReconciliationTaxCutsAndJobsActOf2017TransitionTaxOnAccumulatedForeignEarningsAmount',
'IncomeTaxReconciliationStateAndLocalIncomeTaxes',
'IncomeTaxReconciliationForeignIncomeTaxRateDifferential',
'IncomeTaxReconciliationTaxCredits',
'IncomeTaxReconciliationOtherReconcilingItems'
                ]
sic_code = ['2834' 
         ]

years = ['2020' 
         ]

params = {'fields': ','.join(fields),
         'concept.local-name': ','.join(XBRL_Elements),
         'report.sic-code': ','.join(sic_code),
         'period.fiscal-year': ','.join(years),
         #'fact.offset': 0,
         'period.fiscal-period': 'Y',
         'fact.ultimus': 'TRUE'
         }
# execute the query

search_endpoint = 'https://api.xbrl.us/api/v1/fact/search'
orig_fields = params['fields']

count = 0
while True:
    count +=1
    print ('loop {}'.format(count))
    res = requests.get(search_endpoint, params=params, headers={'Authorization' : 'Bearer {}'.format(access_token)})
    res_json = res.json()
    import urllib
    #print(urllib.parse.unquote(res.url))
    #print(res_json)
    #break
#res_json = res.json(
    #? use offset_value variable here to set offset parameter for next iteration of data
 #                   )
    res_df += (res_json['data'])
    if res_json['paging']['count'] < res_json['paging']['limit']:
        break
    else:
        params['fields'] = orig_fields + ',fact.offset({})'.format(res_json['paging']['limit'])
        print('fields: {}'.format(params['fields']))

# {'limit': 2000, 'offset': 0, 'count': 1000}    
# print the results in a data frame
#print (str(offset_value))
print(pd.DataFrame(res_df))

#print(res_df[1])

# save/append the results to a .csv file
# res_df.append
# res_df.to_csv(filename.csv, sep='\t')




loop 1
fields: period.fiscal-year.sort(DESC),entity.name.sort(ASC),concept.local-name.sort(ASC),fact.value,unit,fact.decimals,report.filing-date,fact.offset(2000)
loop 2
      period.fiscal-year                    entity.name  \
0                   2020        180 Life Sciences Corp.   
1                   2020        180 Life Sciences Corp.   
2                   2020        180 Life Sciences Corp.   
3                   2020        180 Life Sciences Corp.   
4                   2020        180 Life Sciences Corp.   
...                  ...                            ...   
2513                2020                 ZYMEWORKS INC.   
2514                2020  Zynerba Pharmaceuticals, Inc.   
2515                2020  Zynerba Pharmaceuticals, Inc.   
2516                2020  Zynerba Pharmaceuticals, Inc.   
2517                2020  Zynerba Pharmaceuticals, Inc.   

                                     concept.local-name fact.value  unit  \
0            EffectiveIncomeTaxRateContinuing

In [ ]:
# return to queried count above with new value and run query again
new_count = res_json['paging']['count']
print(new_count)
queried_count = new_count

# if the count value is divisible by 2000, copy count value to 